# Visual-language assistant with Ministral-3 and OpenVINO

Ministral-3 (Ministral-3-3B-Instruct-2512) is a lightweight, state-of-the-art multimodal model from [Mistral AI](https://mistral.ai/), combining a 3.4B parameter language model with a 0.4B parameter vision encoder based on the Pixtral architecture. It is designed for efficient visual-language understanding tasks.

**Key Features of Ministral-3:**

* **Multimodal Understanding** — Combines text and vision capabilities in a compact 3B parameter model, enabling image understanding and visual question answering.
* **Long Context Support** — Supports up to 262,144 tokens with YaRN RoPE scaling for extended context processing.
* **Efficient Architecture** — Uses Grouped Query Attention (32 attention heads with 8 KV heads) for memory-efficient inference.
* **Pixtral Vision Encoder** — Employs a PixtralVisionModel with patch-based image processing and multi-modal projection for seamless vision-language integration.

More details about the model can be found in the [model card](https://huggingface.co/mistralai/Ministral-3-3B-Instruct-2512) and the [Mistral AI documentation](https://docs.mistral.ai/).

In this tutorial we consider how to convert and optimize Ministral-3 model for creating a multimodal chatbot. We use [Optimum Intel](https://github.com/huggingface/optimum-intel) for model conversion with [NNCF](https://github.com/openvinotoolkit/nncf) weight compression, and `OVModelForVisualCausalLM` for efficient inference with OpenVINO.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select model](#Select-model)
- [Convert and Optimize model](#Convert-and-Optimize-model)
    - [Select weight format](#Select-weight-format)
- [Prepare OpenVINO Inference Pipeline](#Prepare-OpenVINO-Inference-Pipeline)
    - [Select inference device](#Select-inference-device)
    - [Load OpenVINO model](#Load-OpenVINO-model)
- [Run OpenVINO model inference](#Run-OpenVINO-model-inference)
- [Interactive Demo](#Interactive-Demo)

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO and is using a custom branch of optimum-intel. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/ministral-3/ministral-3.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

Install required packages and setup helper functions.

In [ ]:
%pip uninstall -q -y optimum optimum-intel optimum-onnx
%pip install -q "transformers==5.0.0" "huggingface_hub==1.14.0" "nncf==3.1.0" "torch==2.8" "torchvision==0.23.0" "peft>=0.15.0" "Pillow" "gradio>=4.36,<6" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "openvino>=2025.0.0" "openvino-tokenizers>=2025.0.0"
# Install optimum-intel from PR #1659 which adds Mistral3 (ministral-3) export and inference support
%pip install -q --upgrade-strategy eager "optimum-intel[openvino,nncf] @ git+https://github.com/huggingface/optimum-intel.git@refs/pull/1659/head" --extra-index-url https://download.pytorch.org/whl/cpu

In [ ]:
from pathlib import Path
import requests

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("ministral-3.ipynb")

## Select model
[back to top ⬆️](#Table-of-contents:)

Ministral-3 is available in two sizes. Select the model variant for conversion and inference:

In [ ]:
import ipywidgets as widgets

model_ids = [
    "mistralai/Ministral-3-3B-Instruct-2512-BF16",
    "mistralai/Ministral-3-8B-Instruct-2512-BF16",
]

model_id = widgets.Dropdown(
    options=model_ids,
    value=model_ids[0],
    description="Model:",
)

model_id

In [ ]:
print(f"Selected: {model_id.value}")
pt_model_id = model_id.value
model_dir = Path(pt_model_id.split("/")[-1])

## Convert and Optimize model
[back to top ⬆️](#Table-of-contents:)

Ministral-3 is a PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). For convenience, we will use OpenVINO integration with HuggingFace Optimum.

🤗 [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) is the interface between the 🤗 Transformers and Diffusers libraries and the different tools and libraries provided by Intel to accelerate end-to-end pipelines on Intel architectures.

`optimum-cli` provides command line interface for model conversion and optimization.

General command format:

```bash
optimum-cli export openvino --model <model_id_or_path> --task <task> <output_dir>
```

where `task` is task to export the model for. Additionally, you can specify weights compression using `--weight-format` argument with one of following options: `fp32`, `fp16`, `int8` and `int4`. For int8 and int4, [NNCF](https://github.com/openvinotoolkit/nncf) will be used for weight compression.

### Select weight format
[back to top ⬆️](#Table-of-contents:)

For reducing memory consumption, weights compression optimization can be applied using [NNCF](https://github.com/openvinotoolkit/nncf) and fixed-precision quantization. Weights compression reduces the memory footprint of the model. It can also lead to significant performance improvement for large memory-bound models, such as Large Language Models (LLMs).

In [ ]:
import ipywidgets as widgets

to_compress = widgets.Checkbox(
    value=True,
    description="INT4 weight compression",
    disabled=False,
)

to_compress

In [ ]:
additional_args = {"task": "image-text-to-text"}

if to_compress.value:
    model_export_dir = model_dir / "INT4"
    additional_args.update({"weight-format": "int4"})
else:
    model_export_dir = model_dir / "FP16"
    additional_args.update({"weight-format": "fp16"})

print(f"Model will be exported to: {model_export_dir}")

from cmd_helper import optimum_cli

if not model_export_dir.exists():
    optimum_cli(pt_model_id, model_export_dir, additional_args=additional_args)

## Prepare OpenVINO Inference Pipeline
[back to top ⬆️](#Table-of-contents:)

OpenVINO integration with Optimum Intel provides ready-to-use API for model inference that can be used for smooth integration with transformers-based solutions. For loading model, we will use `OVModelForVisualCausalLM` class that has compatible API with Transformers models and provides the following interface for interaction:

* `from_pretrained` - for loading model from directory.
* `generate` - for running model inference.
* `preprocess_inputs` - for preparing model inputs.

In [ ]:
from optimum.intel.openvino import OVModelForVisualCausalLM
from transformers import AutoProcessor, TextStreamer

### Select inference device
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="AUTO", exclude=["NPU"])

device

### Load OpenVINO model
[back to top ⬆️](#Table-of-contents:)

For model loading we should provide path to model directory and inference device.

In [ ]:
model_export_dir = model_dir / ("INT4" if to_compress.value else "FP16")

# Enable OpenVINO model cache to avoid recompilation after kernel restart
ov_config = {"CACHE_DIR": str(model_export_dir / ".ov_cache")}

model = OVModelForVisualCausalLM.from_pretrained(model_export_dir, device=device.value, ov_config=ov_config)
processor = AutoProcessor.from_pretrained(model_export_dir)

print(f"Model loaded on {device.value}")

## Run OpenVINO model inference
[back to top ⬆️](#Table-of-contents:)

Now, when we have model and processor loaded, we can run model inference.

For preparing input data, we use `preprocess_inputs` method that accepts text and image, and returns model-ready inputs. Additionally, we use `TextStreamer` for streaming token-by-token output.

In [ ]:
from PIL import Image
from io import BytesIO

MAX_IMAGE_SIZE = 512


def load_image(image_file):
    if isinstance(image_file, str) and (image_file.startswith("http") or image_file.startswith("https")):
        response = requests.get(image_file)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_file).convert("RGB")
    # Resize large images to keep patch count manageable
    if max(image.size) > MAX_IMAGE_SIZE:
        image.thumbnail((MAX_IMAGE_SIZE, MAX_IMAGE_SIZE))
    return image


image_url = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
image_file = Path("demo.jpeg")
text_message = "Describe this image."

if not image_file.exists():
    image = load_image(image_url)
    image.save(image_file)
else:
    image = load_image(image_file)

inputs = model.preprocess_inputs(text=text_message, image=image, processor=processor)

In [ ]:
print(f"Question:\n{text_message}")
display(image)
print("Answer:")
model.generate(**inputs, do_sample=False, max_new_tokens=128, streamer=TextStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True))

## Interactive Demo
[back to top ⬆️](#Table-of-contents:)

Now, you can try to chat with the model. Upload an image using the `Upload` button, provide your text message into the `Input` field and click `Submit` to start communication.

In [ ]:
if not Path("gradio_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/ministral-3/gradio_helper.py")
    open("gradio_helper.py", "w").write(r.text)

In [ ]:
from gradio_helper import make_demo

demo = make_demo(model, processor)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/